# Exception Handling in Python

Exceptions are **runtime errors** that can be caught and handled gracefully. Python uses `try/except` blocks for this.

### Why Exception Handling?
- Prevents program crashes
- Provides meaningful error messages
- Allows graceful recovery from errors
- Separates error-handling code from normal logic

> **Prerequisite**: Functions, control flow, basic Python.

## Basic try/except

Syntax:
```python
try:
    # code that might raise an exception
except SomeException:
    # handle the exception
```

In [ ]:
# Basic exception handling
try:
    num = int("hello")  # This will raise ValueError
    print(f"Number: {num}")
except ValueError:
    print("Error: Cannot convert 'hello' to an integer!")

# Catching specific exception and accessing the exception object
try:
    result = 10 / 0
except ZeroDivisionError as e:
    print(f"Math error: {e}")

## Multiple except Blocks

Handle different exception types with different handlers. More specific exceptions should come before general ones.

In [ ]:
# Multiple except blocks
def divide_numbers(a, b):
    """Divide a by b with comprehensive error handling."""
    try:
        result = a / b
    except ZeroDivisionError:
        print("Error: Division by zero!")
    except TypeError as e:
        print(f"Error: Type mismatch — {e}")
    except Exception as e:  # catch-all (use sparingly)
        print(f"Unexpected error: {e}")
    else:
        print(f"Result: {result}")
    finally:
        print("Division attempt completed.")

# Test cases
divide_numbers(10, 2)       # works
divide_numbers(10, 0)       # ZeroDivisionError
divide_numbers(10, "abc")   # TypeError

## else and finally

| Clause | When It Runs |
|--------|-------------|
| `try` | Code that may raise an exception |
| `except` | If an exception was raised |
| `else` | If NO exception was raised (after try) |
| `finally` | **Always** runs (cleanup code) |

In [ ]:
# else and finally demonstration
def safe_divide(a, b):
    """Safely divide two numbers."""
    try:
        result = a / b
    except ZeroDivisionError:
        print("Division by zero!")
        return None
    else:
        print(f"Division successful: {a} / {b} = {result}")
        return result
    finally:
        print("  (finally block — always executes)")

safe_divide(10, 2)
print()
safe_divide(10, 0)

## Raising Exceptions

Use `raise` to explicitly throw an exception. You can raise built-in exceptions or custom ones.

In [ ]:
# Raising built-in exceptions
def withdraw(balance, amount):
    """Withdraw money from account."""
    if amount <= 0:
        raise ValueError("Withdrawal amount must be positive")
    if amount > balance:
        raise ValueError(f"Insufficient funds. Balance: ${balance}")
    return balance - amount

# Test
try:
    new_balance = withdraw(100, 150)
    print(f"New balance: ${new_balance}")
except ValueError as e:
    print(f"Withdrawal failed: {e}")

# Re-raising exceptions
def process_file(filename):
    try:
        with open(filename, 'r') as f:
            return f.read()
    except FileNotFoundError:
        print(f"File '{filename}' not found. Attempting to create it...")
        raise  # re-raise the same exception after logging

## Custom Exception Classes

Create custom exceptions by subclassing `Exception`. This makes error handling more specific.

In [ ]:
# Defining custom exceptions
class InsufficientFundsError(Exception):
    """Raised when account balance is insufficient."""
    pass

class NegativeAmountError(Exception):
    """Raised when amount is negative."""
    pass

class Account:
    """Simple bank account with custom exceptions."""
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance
    
    def deposit(self, amount):
        if amount <= 0:
            raise NegativeAmountError("Deposit amount must be positive")
        self.balance += amount
    
    def withdraw(self, amount):
        if amount <= 0:
            raise NegativeAmountError("Withdrawal amount must be positive")
        if amount > self.balance:
            raise InsufficientFundsError(
                f"Balance: ${self.balance}, Requested: ${amount}"
            )
        self.balance -= amount

# Test
acc = Account("Alice", 500)
try:
    acc.withdraw(600)
except InsufficientFundsError as e:
    print(f"Transaction failed: {e}")
except NegativeAmountError as e:
    print(f"Invalid amount: {e}")
else:
    print(f"Withdrawal successful. Balance: ${acc.balance}")

## Built-in Exception Hierarchy

```
BaseException
 ├── SystemExit
 ├── KeyboardInterrupt
 └── Exception
      ├── ArithmeticError
      │    ├── ZeroDivisionError
      │    └── OverflowError
      ├── LookupError
      │    ├── IndexError
      │    └── KeyError
      ├── ValueError
      ├── TypeError
      ├── FileNotFoundError (OSError)
      ├── AttributeError
      ├── ImportError
      │    └── ModuleNotFoundError
      └── RuntimeError
```

**Rule**: Catch specific exceptions, never bare `except:` (which catches `KeyboardInterrupt` too).

In [ ]:
# Common built-in exceptions
# IndexError
try:
    lst = [1, 2, 3]
    print(lst[10])
except IndexError as e:
    print(f"IndexError: {e}")

# KeyError
try:
    d = {"a": 1}
    print(d["b"])
except KeyError as e:
    print(f"KeyError: {e} — use .get() to avoid this")

# TypeError
try:
    result = "hello" + 5
except TypeError as e:
    print(f"TypeError: {e}")

# AttributeError
try:
    x = None
    x.some_method()
except AttributeError as e:
    print(f"AttributeError: {e}")

## The `assert` Statement

`assert condition, message` — raises `AssertionError` if condition is `False`. Used for debugging and testing (disabled with `-O` flag).

In [ ]:
# Using assert for debugging
def sqrt_positive(x):
    """Return square root (for non-negative numbers only)."""
    assert x >= 0, f"x must be non-negative, got {x}"
    return x ** 0.5

print(f"sqrt(9) = {sqrt_positive(9)}")

try:
    sqrt_positive(-5)
except AssertionError as e:
    print(f"Assertion failed: {e}")

# Assert for input validation in tests
def test_addition():
    assert 2 + 2 == 4, "Basic addition failed"
    assert -1 + 1 == 0
    assert 0 + 0 == 0
    print("All tests passed!")

test_addition()

## Interview Problems

### Problem 1: Safe Division with Retry
Implement a function that divides two numbers with retry logic.

In [ ]:
def safe_divide_with_retry(a, b, max_retries=3):
    """Divide a by b with retry on ZeroDivisionError."""
    for attempt in range(1, max_retries + 1):
        try:
            return a / b
        except ZeroDivisionError as e:
            print(f"Attempt {attempt}: Division by zero")
            if attempt == max_retries:
                raise  # re-raise on last attempt
            b = float(input(f"  Enter a non-zero denominator: "))
    return None

# Simulated test (without interactive input)
try:
    result = safe_divide_with_retry(10, 0, max_retries=2)
    print(f"Result: {result}")
except ZeroDivisionError:
    print("Final error: Division by zero after all retries")

# NOTE: In a real scenario, input() would prompt the user.
# Here we show the pattern; the function will fail without real input.

### Problem 2: Input Validation with Error Handling
Get an integer from the user with validation and retry.

In [ ]:
def get_integer(prompt="Enter an integer: ", min_val=None, max_val=None):
    """Prompt user until a valid integer is entered (within optional range)."""
    while True:
        try:
            user_input = input(prompt)
            value = int(user_input)
            if min_val is not None and value < min_val:
                print(f"Value must be >= {min_val}. Got {value}.")
                continue
            if max_val is not None and value > max_val:
                print(f"Value must be <= {max_val}. Got {value}.")
                continue
            return value
        except ValueError:
            print(f"Invalid input: '{user_input}' is not an integer.")
        except KeyboardInterrupt:
            print("\nInput cancelled.")
            return None

# Test with simulated input
print("Test: get_integer() with simulation")
test_inputs = ["abc", "12.5", "42"]
for test in test_inputs:
    try:
        result = int(test)
        print(f"'{test}' -> {result}")
    except ValueError:
        print(f"'{test}' -> Invalid")

### Problem 3: File Not Found Handler
Try to read a file. If not found, create it with default content.

In [ ]:
import os

def read_or_create_file(filename, default_content=""):
    """Read a file; if not found, create it with default content."""
    try:
        with open(filename, 'r') as f:
            print(f"Reading existing file: {filename}")
            return f.read()
    except FileNotFoundError:
        print(f"File {filename} not found. Creating with default content...")
        with open(filename, 'w') as f:
            f.write(default_content)
        return default_content

# Test
content = read_or_create_file("test_config.txt", "# Default config\nDEBUG=True\n")
print(f"Content:\n{content}")

# Read again — should find the file now
content2 = read_or_create_file("test_config.txt", "")
print(f"Second read:\n{content2}")

# Cleanup
os.remove("test_config.txt")
print("\nCleaned up.")

## Summary

| Concept | Usage |
|---------|-------|
| `try/except` | Catch and handle exceptions |
| `else` | Runs if no exception occurred |
| `finally` | Always runs (cleanup) |
| `raise` | Explicitly throw an exception |
| Custom exceptions | Subclass `Exception` |
| `assert` | Debugging assertions |
| Specific exceptions | Catch `ValueError`, `TypeError`, etc. |

**Next**: Move to **03_Comprehensions_Generators.ipynb** for comprehensions and generators.